In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from matplotlib import pylab as plt
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
import numpy as np
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

In [2]:
path = "diabetes.csv"
data = pd.read_csv(path)
data.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


All of our columns are in numeric data format. Thus we dont need to do any sort of data conversion, or any of the following:

One-hot encoding: turn a categorical column into multiple 0/1 columns (one per category).

Dummy coding: same as one-hot but drop one column (k−1) to avoid multicollinearity.

Indicator/dummy variables: the resulting 0/1 columns.

Label encoding: map categories to integers; for a binary column it’s effectively 0/1.

In [3]:
data.dtypes

Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object

In [4]:

data.isna().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

We might think we have no missing values and all values are numeric. Thus, we should be good to proceed in training our classification model. But, we have some values like Glucose = 0, BP = 0, etc., which should not be possible. They are sort of hidden missing values. Thus, we have to either drop the rows, or place some value there(median of the column).

In [5]:

for index, value in data['Glucose'].items():
    if (value == 0):
        print(data.loc[index])

Pregnancies                  1.00
Glucose                      0.00
BloodPressure               48.00
SkinThickness               20.00
Insulin                      0.00
BMI                         24.70
DiabetesPedigreeFunction     0.14
Age                         22.00
Outcome                      0.00
Name: 75, dtype: float64
Pregnancies                  1.000
Glucose                      0.000
BloodPressure               74.000
SkinThickness               20.000
Insulin                     23.000
BMI                         27.700
DiabetesPedigreeFunction     0.299
Age                         21.000
Outcome                      0.000
Name: 182, dtype: float64
Pregnancies                  1.000
Glucose                      0.000
BloodPressure               68.000
SkinThickness               35.000
Insulin                      0.000
BMI                         32.000
DiabetesPedigreeFunction     0.389
Age                         22.000
Outcome                      0.000
Name: 342, dt

So yes, we do have 0 values which are essentially no data values. Either we can drop those rows or replace the no data value with median. Let's try the latter.

In [6]:
glucose = data['Glucose'].to_numpy()
print(type(glucose))
print(glucose)
median_glucose = np.median(glucose)

<class 'numpy.ndarray'>
[148  85 183  89 137 116  78 115 197 125 110 168 139 189 166 100 118 107
 103 115 126  99 196 119 143 125 147  97 145 117 109 158  88  92 122 103
 138 102  90 111 180 133 106 171 159 180 146  71 103 105 103 101  88 176
 150  73 187 100 146 105  84 133  44 141 114  99 109 109  95 146 100 139
 126 129  79   0  62  95 131 112 113  74  83 101 137 110 106 100 136 107
  80 123  81 134 142 144  92  71  93 122 163 151 125  81  85 126  96 144
  83  95 171 155  89  76 160 146 124  78  97  99 162 111 107 132 113  88
 120 118 117 105 173 122 170  84  96 125 100  93 129 105 128 106 108 108
 154 102  57 106 147  90 136 114 156 153 188 152  99 109  88 163 151 102
 114 100 131 104 148 120 110 111 102 134  87  79  75 179  85 129 143 130
  87 119   0  73 141 194 181 128 109 139 111 123 159 135  85 158 105 107
 109 148 113 138 108  99 103 111 196 162  96 184  81 147 179 140 112 151
 109 125  85 112 177 158 119 142 100  87 101 162 197 117 142 134  79 122
  74 171 181 179 164 104  9

In [7]:
for index, value in data['Glucose'].items():
    if value == 0:
        data['Glucose'].loc[index] = median_glucose


for index, value in data['Glucose'].items():
    if (value == 0):
        print(data.loc[index])

data['Glucose'].value_counts()

/var/folders/vx/0mwc7nyn64x7k2f0jlg9hs300000gn/T/ipykernel_43506/2005483294.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  data['Glucose'].loc[index] = median_glucose
/var/folders/vx/0mwc7nyn64x7k2f0jlg9hs300000gn/T/ipykernel_43506/2005

Glucose
99     17
100    17
117    16
129    14
125    14
       ..
191     1
177     1
44      1
62      1
190     1
Name: count, Length: 135, dtype: int64

Thus we have replaced all the nodata/0 values in the glucose column to the median of that column. Now, we must check and repeat this for all input columns.

In [8]:
#Calculating medians first
BP_median = np.median(data['BloodPressure'].to_numpy())
ST_median = np.median(data['SkinThickness'].to_numpy())
Insulin_median = np.median(data['Insulin'].to_numpy())
BMI_median = np.median(data['BMI'].to_numpy())
DPF_median = np.median(data['DiabetesPedigreeFunction'].to_numpy())
Age_median = np.median(data['Age'].to_numpy())

In [9]:
for index, value in data['BloodPressure'].items():
    if value == 0:
        data['BloodPressure'].loc[index] = BP_median

for index, value in data['SkinThickness'].items():
    if value == 0:
        data['SkinThickness'].loc[index] = ST_median

for index, value in data['Insulin'].items():
    if value == 0:
        data['Insulin'].loc[index] = Insulin_median

for index, value in data['BMI'].items():
    if value == 0:
        data['BMI'].loc[index] = BMI_median

for index, value in data['DiabetesPedigreeFunction'].items():
    if value == 0:
        data['DiabetesPedigreeFunction'].loc[index] = DPF_median

for index, value in data['Age'].items():
    if value == 0:
        data['Age'].loc[index] = Age_median

/var/folders/vx/0mwc7nyn64x7k2f0jlg9hs300000gn/T/ipykernel_43506/2172239989.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  data['BloodPressure'].loc[index] = BP_median
/var/folders/vx/0mwc7nyn64x7k2f0jlg9hs300000gn/T/ipykernel_43506/217

In [10]:
X = data.drop('Outcome', axis = 1)
X.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,30.5,33.6,0.627,50
1,1,85,66,29,30.5,26.6,0.351,31
2,8,183,64,23,30.5,23.3,0.672,32
3,1,89,66,23,94.0,28.1,0.167,21
4,0,137,40,35,168.0,43.1,2.288,33


In [11]:
Y = data['Outcome']
Y.head()

0    1
1    0
2    1
3    0
4    1
Name: Outcome, dtype: int64

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y)

#Lets also directly apply scaling to our feature columns.
s = StandardScaler()
X_train = s.fit_transform(X_train)
X_test = s.transform(X_test)


array([[-0.2425264 , -1.43429825, -0.20534936, ..., -0.01621742,
        -0.59913011,  0.46410908],
       [-0.5482052 ,  0.01255953, -1.0239046 , ..., -0.40968596,
         0.79813284, -0.95784214],
       [-0.85388399, -0.08608986,  1.10433902, ...,  1.89283364,
         1.08258682, -0.37233281],
       ...,
       [ 1.28586756,  0.11120892,  1.92289426, ..., -0.08908197,
        -0.71791309,  1.71877191],
       [-1.15956278,  0.63733902, -0.04163831, ...,  1.42650056,
        -0.80231152, -0.37233281],
       [ 0.67450998,  0.11120892, -0.36906041, ..., -0.38054014,
         0.00728826, -0.12140025]])

In [13]:
lr = LogisticRegression(max_iter=2000)

In [14]:
lr.fit(X_train, Y_train)

LogisticRegression(max_iter=2000)

In [15]:
Y_predicted = lr.predict(X_test)


In [16]:
'''plt.scatter(Y_test,Y_predicted)
plt.xlabel("Actual Charges ($)")
plt.ylabel("Predicted Charges ($)")
plt.title("Actual vs. Predicted Insurance Charges")
plt.plot([min(Y_test), max(Y_test)], [min(Y_test), max(Y_test)], color='red', linestyle='--')
plt.show()
'''
cm =confusion_matrix(y_pred=Y_predicted, y_true=Y_test)
print(cm)
print("Accuracy:", accuracy_score(Y_test, Y_predicted))
print(classification_report(Y_test, Y_predicted))

[[111  14]
 [ 27  40]]
Accuracy: 0.7864583333333334
              precision    recall  f1-score   support

           0       0.80      0.89      0.84       125
           1       0.74      0.60      0.66        67

    accuracy                           0.79       192
   macro avg       0.77      0.74      0.75       192
weighted avg       0.78      0.79      0.78       192



Terrible performance despite decent accuracy. The recall for class 1 is extremely low ~0.56. This implies out of 100 diabetic patients, only 56% of them are classified as diabetic by our classifier. This is detrimental. This happened because our original dataset had less diabetic person. With non diabetic (0) being the more dominant class, our model became lazy and gave more false negatives. 

We have to fix this, and to start off, we can simply use built in parameter "class_weight" and set it to "balanced" such that the classes are more balanced.

In [25]:
lr_v1 = LogisticRegression(max_iter=2000, class_weight='balanced')
lr_v1.fit(X_train, Y_train)
Y_predicted = lr_v1.predict(X_test)
cm =confusion_matrix(y_pred=Y_predicted, y_true=Y_test)
print(cm)
print("Accuracy:", accuracy_score(Y_test, Y_predicted))
print(classification_report(Y_test, Y_predicted))

[[103  24]
 [ 18  47]]
Accuracy: 0.78125
              precision    recall  f1-score   support

           0       0.85      0.81      0.83       127
           1       0.66      0.72      0.69        65

    accuracy                           0.78       192
   macro avg       0.76      0.77      0.76       192
weighted avg       0.79      0.78      0.78       192



There we go. Much balanced recall. But we also see reduced recall for class 0. That means we are now flagging more healthy people as diabetic. This would be less detrimental than not detecting unhealthy patients. This is a decent improvement. But, can we do better?

Lets try improving the features by using polynomials.

In [18]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

lr_v2 = LogisticRegression(max_iter=200000, class_weight='balanced')
lr_v2.fit(X_train_poly, Y_train)
Y_predicted = lr_v2.predict(X_test_poly)
cm =confusion_matrix(y_pred=Y_predicted, y_true=Y_test)
print(cm)
print("Accuracy:", accuracy_score(Y_test, Y_predicted))
print(classification_report(Y_test, Y_predicted))

[[94 31]
 [19 48]]
Accuracy: 0.7395833333333334
              precision    recall  f1-score   support

           0       0.83      0.75      0.79       125
           1       0.61      0.72      0.66        67

    accuracy                           0.74       192
   macro avg       0.72      0.73      0.72       192
weighted avg       0.75      0.74      0.74       192



No avail. Lets try a more complex model based on K-NN

In [19]:
from sklearn.neighbors import KNeighborsClassifier

KNN_v3 = KNeighborsClassifier(n_neighbors=5)
KNN_v3.fit(X_train, Y_train)
Y_predicted = KNN_v3.predict(X_test)
cm =confusion_matrix(y_pred=Y_predicted, y_true=Y_test)
print(cm)
print("Accuracy:", accuracy_score(Y_test, Y_predicted))
print(classification_report(Y_test, Y_predicted))

[[108  17]
 [ 26  41]]
Accuracy: 0.7760416666666666
              precision    recall  f1-score   support

           0       0.81      0.86      0.83       125
           1       0.71      0.61      0.66        67

    accuracy                           0.78       192
   macro avg       0.76      0.74      0.74       192
weighted avg       0.77      0.78      0.77       192



It actually did worse.

In [20]:
from sklearn.tree import DecisionTreeClassifier
DT_v4 = DecisionTreeClassifier(class_weight='balanced')
DT_v4.fit(X_train, Y_train)
Y_predicted = DT_v4.predict(X_test)
cm =confusion_matrix(y_pred=Y_predicted, y_true=Y_test)
print(cm)
print("Accuracy:", accuracy_score(Y_test, Y_predicted))
print(classification_report(Y_test, Y_predicted))

[[96 29]
 [25 42]]
Accuracy: 0.71875
              precision    recall  f1-score   support

           0       0.79      0.77      0.78       125
           1       0.59      0.63      0.61        67

    accuracy                           0.72       192
   macro avg       0.69      0.70      0.69       192
weighted avg       0.72      0.72      0.72       192



Still worse.

In [26]:
from sklearn.ensemble import RandomForestClassifier
RF_v5 = RandomForestClassifier(class_weight='balanced')
RF_v5.fit(X_train, Y_train)
Y_predicted = RF_v5.predict(X_test)
cm =confusion_matrix(y_pred=Y_predicted, y_true=Y_test)
print(cm)
print("Accuracy:", accuracy_score(Y_test, Y_predicted))
print(classification_report(Y_test, Y_predicted))

[[114  13]
 [ 24  41]]
Accuracy: 0.8072916666666666
              precision    recall  f1-score   support

           0       0.83      0.90      0.86       127
           1       0.76      0.63      0.69        65

    accuracy                           0.81       192
   macro avg       0.79      0.76      0.77       192
weighted avg       0.80      0.81      0.80       192

